In [1]:
import numpy as np
import pandas as pd

url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI',
           'DiabetesPedigreeFunction', 'Age', 'Outcome']

df = pd.read_csv(url, names=columns)

df.shape
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [2]:
print(df.isnull().sum())

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


In [3]:
print(df.describe())

       Pregnancies     Glucose  BloodPressure  SkinThickness     Insulin  \
count   768.000000  768.000000     768.000000     768.000000  768.000000   
mean      3.845052  120.894531      69.105469      20.536458   79.799479   
std       3.369578   31.972618      19.355807      15.952218  115.244002   
min       0.000000    0.000000       0.000000       0.000000    0.000000   
25%       1.000000   99.000000      62.000000       0.000000    0.000000   
50%       3.000000  117.000000      72.000000      23.000000   30.500000   
75%       6.000000  140.250000      80.000000      32.000000  127.250000   
max      17.000000  199.000000     122.000000      99.000000  846.000000   

              BMI  DiabetesPedigreeFunction         Age     Outcome  
count  768.000000                768.000000  768.000000  768.000000  
mean    31.992578                  0.471876   33.240885    0.348958  
std      7.884160                  0.331329   11.760232    0.476951  
min      0.000000                  

In [4]:
cols_with_missing_values = ['Glucose','BloodPressure','SkinThickness','Insulin','BMI']
df[cols_with_missing_values] = df[cols_with_missing_values].replace(0, np.nan)

print(df.isnull().sum())

Pregnancies                   0
Glucose                       5
BloodPressure                35
SkinThickness               227
Insulin                     374
BMI                          11
DiabetesPedigreeFunction      0
Age                           0
Outcome                       0
dtype: int64


In [5]:
df.fillna(df.mean(), inplace=True)

print(df.isnull().sum())

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


In [6]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X  = df.drop('Outcome', axis=1)
y = df['Outcome']

X_train,X_test,y_train,y_test = train_test_split(X,y, test_size=0.3, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(X_train_scaled.shape)
print(X_test_scaled.shape)

(537, 8)
(231, 8)


In [8]:
X_train_scaled[0]

array([-0.8362943 , -0.89610788, -1.00440048, -1.27450178, -1.14686808,
       -1.20403257, -0.61421636, -0.94861028])

In [9]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

baseline_model = RandomForestClassifier(random_state=42)
baseline_model.fit(X_train_scaled,y_train)
y_pred_baseline = baseline_model.predict(X_test_scaled)
baseline_accuracy = accuracy_score(y_test, y_pred_baseline)

print(f"Accuracy: {baseline_accuracy * 100:.2f}%\n")
print(classification_report(y_test,y_pred_baseline))


Accuracy: 74.89%

              precision    recall  f1-score   support

           0       0.81      0.81      0.81       151
           1       0.64      0.64      0.64        80

    accuracy                           0.75       231
   macro avg       0.72      0.72      0.72       231
weighted avg       0.75      0.75      0.75       231



In [13]:
import optuna
from sklearn.model_selection import cross_val_score

def objective(trial):
    n_estimators = trial.suggest_int('n_estimators', 50, 300)
    max_depth = trial.suggest_int('max_depth',3, 20)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 10)

    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        random_state=42
    )

    score = cross_val_score(model, X_train_scaled, y_train, cv=3, scoring='accuracy').mean()
    return score


optuna.logging.set_verbosity(optuna.logging.WARNING)

study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler())
study.optimize(objective, n_trials=50)

print(study.best_trial.params)


best_rf_model = RandomForestClassifier(**study.best_trial.params, random_state=42)
best_rf_model.fit(X_train_scaled, y_train)
y_pred_optuna = best_rf_model.predict(X_test_scaled)
optuna_accuracy = accuracy_score(y_test, y_pred_optuna)
print(f"Baseline Accuracy : {baseline_accuracy * 100:.2f}%")
print(f"Optuna Accuracy : {optuna_accuracy * 100:.2f}%")

{'n_estimators': 69, 'max_depth': 18, 'min_samples_split': 2}
Baseline Accuracy : 74.89%
Optuna Accuracy : 75.32%


## Samplers

In [14]:
random_study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.RandomSampler(seed=42)
)

random_study.optimize(objective, n_trials=50)

print(random_study.best_trial.params)

random_model = RandomForestClassifier(**random_study.best_trial.params, random_state=42)
random_model.fit(X_train_scaled, y_train)
random_accuracy = accuracy_score(y_test, random_model.predict(X_test_scaled))

print(f"TPE Accuracy   : {optuna_accuracy * 100:.2f}%")
print(f"Random Accuracy: {random_accuracy * 100:.2f}%")

{'n_estimators': 200, 'max_depth': 19, 'min_samples_split': 2}
TPE Accuracy   : 75.32%
Random Accuracy: 74.03%


## visualize

In [17]:
from optuna.visualization import plot_optimization_history, plot_param_importances, plot_slice

fig1 = plot_optimization_history(study)
fig1.show()


In [18]:
fig2 = plot_param_importances(study)
fig2.show()


In [19]:
fig3 = plot_slice(study)
fig3.show()

## Train multiple model at once

In [20]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

def objective(trial):
    classifier_name = trial.suggest_categorical('classifier',['SVM','RandomForest','GradientBoosting'])

    if classifier_name == 'SVM':
        c = trial.suggest_float('C', 0.1, 10.0, log=True)
        kernel = trial.suggest_categorical('kernel', ['linear', 'rbf'])
        model = SVC(C=c, kernel=kernel, random_state=42)

    elif classifier_name == 'RandomForest':
        n_estimators = trial.suggest_int('rf_n_estimators', 50, 200)
        max_depth = trial.suggest_int('rf_max_depth', 3, 10)
        model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)

    elif classifier_name == 'GradientBoosting':
        n_estimators = trial.suggest_int('gb_n_estimators', 50, 200)
        learning_rate = trial.suggest_float('gb_learning_rate', 0.01, 0.2, log=True)
        model = GradientBoostingClassifier(n_estimators=n_estimators, learning_rate=learning_rate, random_state=42)

    score = cross_val_score(model, X_train_scaled, y_train, cv=3, scoring='accuracy').mean()
    return score

multi_study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
multi_study.optimize(objective,n_trials=40)

print('Best model and its parameters:', multi_study.best_trial.params)
print(f"Best acuracy: {multi_study.best_trial.value * 100:.2f}%")


Best model and its parameters: {'classifier': 'SVM', 'C': 0.11672711987380449, 'kernel': 'linear'}
Best acuracy: 78.96%


In [21]:
import xgboost as xgb
import optuna

def objective_xgb(trial):

    param = {
        'verbosity': 0,
        'objective': 'binary:logistic', 
        'eval_metric': 'logloss',       
        'booster': 'gbtree',
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 9),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0)
    }

    dtrain = xgb.DMatrix(X_train_scaled, label=y_train)
    dtest = xgb.DMatrix(X_test_scaled, label=y_test)

   
    pruning_callback = optuna.integration.XGBoostPruningCallback(trial, "eval-logloss")

   
    bst = xgb.train(
        param,
        dtrain,
        num_boost_round=100,
        evals=[(dtest, "eval")], 
        callbacks=[pruning_callback], # Pruner attach kar diya!
        verbose_eval=False
    )

    # Predictions
    preds = bst.predict(dtest)
    pred_labels = np.round(preds) 
    accuracy = accuracy_score(y_test, pred_labels)
    
    return accuracy


study_xgb = optuna.create_study(
    direction='maximize', 
    pruner=optuna.pruners.SuccessiveHalvingPruner()
)
study_xgb.optimize(objective_xgb, n_trials=30)


print("Best Parameters:", study_xgb.best_trial.params)
print(f"Best Accuracy: {study_xgb.best_trial.value * 100:.2f}%")

pruned_trials = [t for t in study_xgb.trials if t.state == optuna.trial.TrialState.PRUNED]
print(f"\nTotal Trials: 30")
print(f"Pruned Trials: {len(pruned_trials)}")

c:\Users\husna\AppData\Local\Programs\Python\Python312\Lib\site-packages\optuna\integration\xgboost.py:14: FutureWarning: `optuna.integration.xgboost` has been deprecated in v4.9.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v4.9.0. Use `optuna_integration.xgboost` instead.
  optuna_warn(f"{msg} Use `optuna_integration.xgboost` instead.", FutureWarning)


Best Parameters: {'learning_rate': 0.05452599251830121, 'max_depth': 8, 'subsample': 0.5452113993739065}
Best Accuracy: 74.03%

Total Trials: 30
Pruned Trials: 28


In [22]:
from optuna.visualization import plot_intermediate_values, plot_optimization_history

plot_intermediate_values(study_xgb).show()

plot_optimization_history(study_xgb).show()